In [ ]:
pip install findspark

In [ ]:
pip install pyspark pandas

In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [ ]:
import os
import findspark
#from pyspark import SparkConf, SparkContext
#from pyspark.sql import SparkSession
#import pyspark
import time
#from pyspark.sql import functions as sf

# --- Set the SPARK_HOME environment variable manually ---
os.environ["SPARK_HOME"] = "C:/spark" 
findspark.init()
print("Findspark initialized successfully.") 

In [ ]:
import os
import findspark
import sqlite3
#import pandas as pd
import time
from pyspark.sql import SparkSession

# --- Set Environment Variables (Your setup) ---
os.environ["JAVA_HOME"] = r"C:\Program Files\BellSoft\LibericaJDK-17" 
os.environ["SPARK_HOME"] = r"C:\spark" 
os.environ["HADOOP_HOME"] = r"C:\Hadoop" 

findspark.init()
print("Findspark initialized and environment variables set.")

# --- Configuration for Spark Session (Your setup) ---
# NOTE: Check the extension of the JDBC driver path: mssql-jdbc-13.2.1.jre11.jarr (extra r)
mssql_jdbc_path = r"C:\spark_drivers\mssql-jdbc-13.2.1.jre11.jar" # Corrected path
spark_connector_path = r"C:\spark_drivers\spark-mssql-connector_2_12-1.0.1.jar"  # no need for this program actually
sqlite_jdbc_path = r"C:\spark_drivers\sqlite-jdbc-3.44.1.0.jar" # <-- Download this JAR
# Combine all jars for the Spark Session
all_jars = f"{mssql_jdbc_path},{spark_connector_path},{sqlite_jdbc_path}"

sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=AdventureSales;encrypt=true;trustServerCertificate=true"
sqlserver_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchSize": "1000"
}
source_view_name = "sttwk1a"

# --- SQLite Configuration ---
sqlite_db_file_path = r"C:\sqlite\AdventureSales.db" # Destination SQLite DB file path
sqlite_table_name = "sttwk" # Destination table name in SQLite DB
# --- SQLite Configuration (New approach requires an absolute URL) ---
# We need the full JDBC URL for the destination SQLite file path
# SQLite JDBC URL format uses 'jdbc:sqlite:' followed by the file path
sqlite_url = f"jdbc:sqlite:{sqlite_db_file_path}"



sqlite_properties = {
    "driver": "org.sqlite.JDBC",
    "batchsize": "20000", # Suggest batching inserts
    "isolationLevel": "NONE" # SQLite doesn't support standard transaction isolation levels
}


# Define the SQLite CREATE TABLE statement needed to match your source data

create_table_sql = """
CREATE TABLE IF NOT EXISTS sttwk (
    branch_id CHAR(15) NOT NULL,
    sp_id VARCHAR(50) NOT NULL,
    custd_id VARCHAR(50) NOT NULL,
    prod_id CHAR(20) NOT NULL,
    sls_id CHAR(15) NOT NULL,
    year CHAR(4) NOT NULL,
    month CHAR(2) NOT NULL,
    week CHAR(2) NOT NULL,
    net_qty NUMERIC(12, 4) NULL,
    net_amt NUMERIC(18, 4) NULL,
    grs_qty NUMERIC(12, 4) NULL,
    grs_amt NUMERIC(18, 4) NULL,
    rtn_qty NUMERIC(10, 2) NULL,
    rtn_amt NUMERIC(15, 2) NULL
);
"""

# --- Create Spark Session ---
print("Attempting to create Spark Session...")
spark = SparkSession.builder \
    .appName("SQLServerToSQLiteTransfer_LocalJars") \
    .config("spark.jars", all_jars) \
    .getOrCreate()
    
print(f"The active Spark version is: {spark.version}")
print("Spark session created successfully.")

start_time = time.time()
# --- Step 1: Read data from SQL Server into Spark DataFrame ---
print(f"Reading data from SQL Server view: {source_view_name}...")
df_source = spark.read.jdbc(
    url=sql_server_url,
    table=source_view_name,
    properties=sqlserver_properties
)
# Optional: display schema to confirm data types
df_source.printSchema() 
record_count = df_source.count()
print(f"Total rows read from SQL Server: {record_count}")
end_time = time.time()
print(f"Data read complete. {record_count} rows read in {end_time - start_time:.2f} seconds.")


In [ ]:

# --- Step 2: Create the SQLite database file and table structure ---
all_jars_paths_final = f"{all_jars},{sqlite_jdbc_path}"

print(f"Starting optimized data write to SQLite table '{sqlite_table_name}'...")

print(f"Starting data insertion into SQLite table '{sqlite_table_name}'...")
start_time = time.time()
# Spark manages the parallel, batched insert using the SQLite JDBC driver
df_source.write.jdbc(
    url=sqlite_url,
    table=sqlite_table_name,
    mode="overwrite", # Creates the table fresh every time
    properties=sqlite_properties
)

end_time = time.time()
print(f"Data transfer complete. {record_count} rows inserted in {end_time - start_time:.2f} seconds.")

print(f"Data transfer complete. {record_count} rows inserted in {end_time - start_time:.2f} seconds.")

# --- Cleanup ---
spark.stop()
print("Connections closed. Program finished.")

In [ ]:
# Drop the temporary 'year_week_int' column before writing to Oracle
# just do this cell if dont want to include them !!!!
#df_to_write = df_partitioned.drop("year_week_int")
